In [21]:
import pandas as pd
import json
import csv

In [26]:
file_path = "/Users/kanhaiwadekar/Downloads/yelp_academic_dataset_business.json"

# read in business data and convert to df
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

# filter for businesses in CA 
ca_df = df[df['state'] == 'CA']

# filter for Restaurants 
restaurants_ca = ca_df[
    ca_df['categories'].notna() & 
    ca_df['categories'].str.contains('Restaurants')
]

# remove duplicates by keeping the 1st one only (like chains)
restaurants_ca = restaurants_ca.drop_duplicates(subset='name', keep='first')

restaurants_ca = restaurants_ca.reset_index(drop=True)
business_ids = set(restaurants_ca['business_id']) 

print(f"Number of Entries: {len(restaurants_ca)}")
print(restaurants_ca.head())

Number of Entries: 1035
              business_id                             name                    address           city state postal_code   latitude   longitude  stars  review_count  is_open  \
0  IDtLPgUrqorrpqSLdfMhZQ             Helena Avenue Bakery      131 Anacapa St, Ste C  Santa Barbara    CA       93101  34.414445 -119.690672    4.0           389        1   
1  SZU9c8V2GuREDN5KgyHFJw  Santa Barbara Shellfish Company          230 Stearns Wharf  Santa Barbara    CA       93101  34.408715 -119.685019    4.0          2404        1   
2  4xhGQGdGqU60BIznBjqnuA     California Tacos and Taproom  956 Embarcadero Del Norte     Isla Vista    CA       93117  34.411555 -119.855077    4.0            49        0   
3  ifjluUv4VASwmFqEp8cWlQ                    Marty's Pizza         2733 De La Vina St  Santa Barbara    CA       93105  34.436236 -119.726147    4.0            64        1   
4  VeFfrEZ4iWaecrQg6Eq4cg                         Cal Taco  7320 Hollister Ave, Ste 1         Goleta 

In [16]:
reviews_json_path = "/Users/kanhaiwadekar/Downloads/yelp_academic_dataset_review.json"
output_csv_template = "ca_restaurant_reviews_part{}.csv"

fields = ["review_id", "user_id", "business_id", "stars", "date", "text", "useful", "funny", "cool"]

#count total reviews for the businesses
total_reviews = 0
with open(reviews_json_path, 'r') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in business_ids:
            total_reviews += 1

chunk_size = total_reviews // 4  # roughly equal per file

print(f"Total reviews: {total_reviews}, chunk size: {chunk_size}")

# write to 4 separate CSVs
count = 0
file_index = 1
out = open(output_csv_template.format(file_index), 'w', newline='', encoding='utf-8')
writer = csv.DictWriter(out, fieldnames=fields)
writer.writeheader()

with open(reviews_json_path, 'r') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in business_ids:
            writer.writerow(review)
            count += 1
            
            # Switch to next file when reaching chunk_size
            if count >= file_index * chunk_size and file_index < 4:
                out.close()
                file_index += 1
                out = open(output_csv_template.format(file_index), 'w', newline='', encoding='utf-8')
                writer = csv.DictWriter(out, fieldnames=fields)
                writer.writeheader()
                
            if count % 100000 == 0:
                print(f"Wrote {count:,} reviews")

out.close()
print("Done!")

Total reviews: 190483, chunk size: 47620
Wrote 100,000 reviews
Done!


In [32]:
csv1 = pd.read_csv("ca_restaurant_reviews_with_sentiment1.csv")
csv2 = pd.read_csv("ca_restaurant_reviews_with_sentiment2.csv")
combined = pd.concat([csv1, csv2],ignore_index=True)
combined.to_csv("ca_restaurant_reviews_with_sentiment_combined.csv", index=False)
combined = pd.read_csv("ca_restaurant_reviews_with_sentiment_combined.csv")

# Truncate the text column for display
combined['text_short'] = combined['text'].str.slice(0, 100)  # first 100 chars

# Set display options
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', 1500)         # make wide enough for one line
pd.set_option('display.max_colwidth', 100)   # show up to 120 chars per column

# Print with truncated text
print(f"Number of Reviews: {len(combined)}")
print(combined.head()[[
    'review_id', 'user_id', 'business_id', 'stars', 'food_sentiment_score', 'text_short'
]])

Number of Reviews: 190483
                review_id                 user_id             business_id  stars  food_sentiment_score                                                                                           text_short
0  pUycOfUwM8vqX7KjRRhUEA  59MxRhNVhU9MYndMkz0wtw  gebiRewfieSdtt17PTW6Zg    3.0                     6  Had a party of 6 here for hibachi. Our waitress brought our separate sushi orders on one plate s...
1  eCiWBf1CJ0Zdv1uVarEhhw  OhECKhQEexFypOMY6kypRw  vC2qm1y3Au5czBtbhc-DNw    4.0                     9  Yes, this is the only sushi place in town. However, it is great when you're craving sushi and do...
2  YbMyvlDA2W3Py5lTz8VK-A  4hBhtCSgoxkrFgHa4YAD-w  bbEXAEFr4RYHLlZ-HFssTA    5.0                     7  Great burgers,fries and salad!  Burgers have a hint of salt and pepper flavor.\n\nThis location ...
3  L0jv8c2FbpWSlfNC6bbUEA  bFPdtzu11Oi0f92EAcjqmg  IDtLPgUrqorrpqSLdfMhZQ    5.0                     7  What a great addition to the Funk Zone!  Grab a bite, 